# Geographic Attribution of MCL Surface Names Using a Hybrid Inference Model

# Introduction

The Meta Content Library (MCL) provides structured access to a vast collection of public Facebook and Instagram content, including posts, link attachments, surface metadata, engagement statistics, and associated entity descriptors.
Among these fields, the **`surface.name`** attribute is especially important, as it represents the name of the source surface attached to a given piece of content—often corresponding to a news outlet, organization, media source, page name, or other public-facing entity.

However, `surface.name` entries in the MCL dataset are:

* Highly diverse (spanning thousands of regions and languages)
* Often multilingual
* Frequently informal, user-generated, or brand-specific
* Occasionally noisy, misspelled, or partially corrupted
* Not standardized or associated with explicit geo-identifiers

Because the MCL does not provide explicit country-level information for `surface.name`, downstream analyses—such as regional narrative tracking, outbreak-related discourse assessment, or cross-country risk comparisons—require a systematic way to **infer the most likely country of origin** for each surface name.

This notebook builds a reproducible pipeline to perform this inference using a combination of keyword matching, language inference, name-origin heuristics, and semantic similarity.

---

# Objectives

The purpose of this notebook is to develop an **accurate, scalable, and explainable country-inference system** tailored specifically to the structure and characteristics of Meta Content Library data.
By the end of the workflow, each row in the MCL dataset will include two new fields:

* **`inferred_country`**
* **`inferred_country_confidence`**

To accomplish this, the notebook will:

### **1. Load and explore MCL `surface.name` values**

* Inspect coverage, duplication, and noise patterns
* Identify multilingual and non-standard naming behaviors common in MCL

### **2. Build a hybrid inference model tailored to MCL**

* **Keyword-based classification** using expanded global + Asian + augmented dictionaries
* **Language detection** applied to the text of surface names
* **Name-origin inference** for personal-name surface names (common in MCL influencer or personality pages)
* **Embedding-based semantic similarity** to map names close to known country-labeled exemplars

### **3. Combine all inference signals into an ensemble predictor**

* Weighted scoring logic reflecting reliability of each signal
* Final country selection + confidence estimation

### **4. Diagnose and refine the “Unknown” bucket**

* Identify why MCL surfaces fail classification
* Detect patterns such as:

  * Non-geographic brands
  * Duplicate or noisy MCL entries
  * Misspellings and OCR artifacts
* Automatically mine new candidate keywords to improve future classification

### **5. Join inferred results back into the full MCL dataset**

* Merge on `surface.name`
* Append `inferred_country` + confidence fields
* Save the enriched dataset for downstream modeling

### **6. Export the enriched dataset**

* Produce a final CSV suitable for:

  * Country-level discourse aggregation
  * Geospatial narrative mapping
  * Regional trend analysis
  * Epidemiological early warning signals
  * Misinformation risk scoring


In [16]:
import pandas as pd
import spacy
from langdetect import detect
from sentence_transformers import SentenceTransformer, util
import re


#####################################################################
# 1. LOAD MODELS
#####################################################################

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

print("Loading Sentence-BERT embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading spaCy model...


/opt/homebrew/lib/python3.11/site-packages/spacy/util.py:922: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.7). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Loading Sentence-BERT embeddings...


### Data Load and Surface Name Extraction

This section loads the primary Meta Content Library dataset and prepares the list of unique surface names for downstream country-inference analysis.

1. **Load the main MCL dataset**
   The `h5n1-100k-2017–2025.csv` file is read into `df_main`, which contains all post-level records, including metadata fields such as `surface.name`.

2. **Generate a cleaned text column**
   A new field, `clean_text`, is created by:

   * Converting text to lowercase
   * Removing URLs, mentions, hashtags, and other non-alphanumeric patterns
     This ensures that downstream NLP and classification steps operate on normalized content.

3. **Extract unique surface names**
   The script identifies all distinct values from the `surface.name` column and saves them to a separate CSV (`unique_surface_names.csv`).
   These values represent the set of all source entities in the dataset and are the input for the country-inference pipeline.

4. **Reload unique surface names for processing**
   The notebook reads back the saved file and prepares the `names` series (filling missing entries with an empty string). This ensures that the country-inference model operates only on the deduplicated set of surface names.

This preprocessing stage lays the foundation for the next steps, where each surface name will be enriched with inferred geographic metadata.


In [20]:
df_main = pd.read_csv("./data/h5n1-100k-2017-2025.csv")

df_main['clean_text'] = (
    df_main['text']
    .astype(str)
    .apply(lambda x: re.sub(r"http\S+|www\S+|@\S+|#\S+", "", x.lower()))
)

df_surfacenames = df_main["surface.name"].drop_duplicates()
df_surfacenames.to_csv("./data/unique_surface_names.csv")

df = pd.read_csv("unique_surface_names.csv")
names = df["surface.name"].fillna("")



## Language + Keyword Configuration for Country Inference

This section defines the foundational linguistic resources used by the country-inference model. These resources provide the rule-based components that help classify `surface.name` values into likely country origins.

### Language Detection Helper**

A lightweight wrapper function, `detect_language()`, uses `langdetect` to identify the probable language of each surface name.
If detection fails (due to noise, numbers, corrupted strings), the function returns `"unknown"` safely.

Language detection later contributes to the ensemble’s scoring logic—for example:

* `en` → potentially U.S., U.K., Canada, Australia
* `tl` → Philippines
* `vi` → Vietnam
* etc.

---

### Base Global + Asian Keyword Dictionary**

The `COUNTRY_KEYWORDS` dictionary defines a curated set of country-specific keywords drawn from:

* **North America & Europe**
* **East Asia (China, Japan, Korea, etc.)**
* **South Asia (India, Pakistan, Bangladesh, etc.)**
* **Southeast Asia (Indonesia, Malaysia, Philippines, etc.)**|
* **Middle East (Saudi Arabia, UAE, Iraq, etc.)**

These keywords include:

* Country names
* Demonyms
* City names
* Language markers
* Media-specific terms
* Common character sets (e.g., Chinese Hanzi, Japanese Kanji)

This forms the *core* of the rule-based country-inference logic.

---

### Augmented Keyword Dictionary (User-Provided Enhancements)**

The `AUGMENTED_KEYWORDS` dictionary incorporates domain-specific or dataset-specific additional patterns identified during exploration of Meta Content Library surface names.

These include:

* U.S.-specific geographical markers (e.g., *Wisconsin, Illinois, La Crosse*)
* News/media identifiers (e.g., *bbc, sky news*)
* Local language variants (e.g., *pilipinas, ph* for the Philippines)
* Institutional names (e.g., *center for infectious disease*)
* Regional spellings or misspellings commonly present in MCL data

These keywords significantly improve coverage for ambiguous sources or noisy text strings.

---

### Merging Augmented Keywords Into Base Dictionary**

This final section integrates `AUGMENTED_KEYWORDS` into the main `COUNTRY_KEYWORDS` dictionary:

* If a country already exists in the base dictionary, its keyword list is **extended**.
* If it does not exist, a new entry is **created**.

This merging operation ensures that:

* The final keyword set is comprehensive
* No country loses previously defined patterns
* Dataset-specific enhancements are fully incorporated

The resulting enriched dictionary is used downstream in the `keyword_country()` function to classify each surface name.

---


In [21]:
# Language detection

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

In [22]:
# Base Global with Asian Keywords augments

COUNTRY_KEYWORDS = {
    # --- USA / Europe ---
    "United States": ["american", "usa", "u.s.", "california", "new york", "texas"],
    "United Kingdom": ["uk", "british", "wales", "england", "london"],
    "Germany": ["german", "deutsch", "und", "mehr", "berlin"],
    "France": ["france", "french", "français", "paris"],
    "Spain": ["spain", "spanish", "españa", "madrid"],
    "Italy": ["italy", "italian", "roma"],
    "Netherlands": ["dutch", "nederland", "amsterdam"],
    "Romania": ["romania", "romanian"],
    "Canada": ["canada", "canadian", "toronto"],
    "Australia": ["australia", "aussie"],

    # --- East Asia ---
    "China": ["china", "chinese", "北京", "上海", "广州", "日报", "新闻"],
    "Japan": ["japan", "japanese", "東京", "新聞", "朝日", "nhk"],
    "South Korea": ["korea", "한국", "서울", "일보", "신문"],
    "North Korea": ["dprk", "pyongyang"],
    "Taiwan": ["taiwan", "台灣", "台北"],
    "Mongolia": ["mongolia"],

    # --- South Asia ---
    "India": ["india", "indian", "delhi", "hindustan", "bharat"],
    "Pakistan": ["pakistan", "karachi", "lahore", "islamabad"],
    "Bangladesh": ["bangladesh", "dhaka", "বাংলা"],
    "Sri Lanka": ["sri lanka", "colombo"],
    "Nepal": ["nepal", "kathmandu"],
    "Bhutan": ["bhutan"],
    "Maldives": ["maldives"],

    # --- SE Asia ---
    "Indonesia": ["indonesia", "jakarta"],
    "Malaysia": ["malaysia", "kuala lumpur"],
    "Singapore": ["singapore"],
    "Philippines": ["philippines", "manila"],
    "Thailand": ["thailand", "bangkok"],
    "Vietnam": ["vietnam", "việt"],
    "Laos": ["laos"],
    "Cambodia": ["cambodia", "khmer"],
    "Myanmar": ["myanmar", "burma"],
    "Brunei": ["brunei"],

    # --- Middle East ---
    "Turkey": ["turkey", "istanbul"],
    "Saudi Arabia": ["saudi", "riyadh"],
    "UAE": ["uae", "dubai"],
    "Israel": ["israel", "jerusalem"],
    "Iran": ["iran", "tehran"],
    "Iraq": ["iraq", "baghdad"],
    "Qatar": ["qatar", "doha"],
    "Kuwait": ["kuwait"],
    "Jordan": ["jordan", "amman"],
    "Lebanon": ["lebanon", "beirut"],
    "Oman": ["oman"],
    "Bahrain": ["bahrain"],
    "Yemen": ["yemen"],
}


# Augmented keywords

AUGMENTED_KEYWORDS = {
    "United States": [
        "united states", "usa", "cdc", "la crosse", "american", "madison",
        "center for infectious disease", "wisconsin", "new jersey", "iowa",
        "sci-tech", "shalom wildlife sanctuary", "mriglobal", "minnesota", "illinois",
        "center for infectious fisease research"
    ],
    "United Kingdom": ["uk", "britain", "bbc", "sky news"],
    "India": ["india", "hindu", "times of india", "haryana"],
    "Malaysia": ["malaysia"],
    "Philippines": ["philippines", "ph", "pilipinas"],
    "Australia": ["australia", "parks victoria"],
    "France": ["france", "le monde"],
    "China": ["china"],
    "Japan": ["japan", "nhk"],
    "Indonesia": ["indonesia"],
    "Ireland": ["cork city council"],
    "Canada": ["bc", "toronto"],
    "Egypt": ["egypt"],
    "Thailand": ["thailand"],
    "South Africa": ["mossel"],
}

In [23]:
# Merge Augments into the base

for country, keywords in AUGMENTED_KEYWORDS.items():
    if country in COUNTRY_KEYWORDS:
        COUNTRY_KEYWORDS[country].extend(keywords)
    else:
        COUNTRY_KEYWORDS[country] = keywords

Here is a polished and technically clear **Markdown description** for the cell you provided. It explains the purpose of each function and how they contribute to the hybrid inference model.

---

## Location Extraction, Semantic Similarity, and Keyword Matching

This section defines three core components of the hybrid country-inference pipeline. Each component provides a different type of signal—entity recognition, semantic proximity, or explicit string cues—that will later be combined in the ensemble scoring model.

---

### **7. Named Entity Recognition (NER) – Location Extraction**

```python
def ner_location(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
```

This function uses spaCy’s NER model to extract **geopolitical entities (GPE)** and **locations (LOC)** from a surface name string.
For example:

* `"Tokyo Shimbun"` → `["Tokyo"]`
* `"The Brecon and Radnor Express"` → `["Brecon", "Radnor"]`

These extracted location tokens are later fed into the semantic embedding model to infer the closest matching country.

---

### **8. Embedding Similarity Model**

```python
COUNTRY_LIST = list(COUNTRY_KEYWORDS.keys())
country_emb = model.encode(COUNTRY_LIST, convert_to_tensor=True)
```

A sentence-transformer model encodes all country names into vector embeddings.
These serve as reference points for similarity comparison.

```python
def embedding_country(text):
    text_emb = model.encode(text, convert_to_tensor=True)
    scores = util.cos_sim(text_emb, country_emb)[0]
    best_idx = scores.argmax().item()
    return COUNTRY_LIST[best_idx], float(scores[best_idx])
```

The surface name (or extracted location token) is embedded and compared to all country embeddings using cosine similarity.
The model returns:

* The country with the highest similarity
* The associated similarity score

This provides a **semantic fallback mechanism** when keywords or language detection are inconclusive.

---

### **9. Keyword Matching**

```python
def keyword_country(text):
    txt = text.lower()
    for country, keywords in COUNTRY_KEYWORDS.items():
        for kw in keywords:
            if kw.lower() in txt:
                return country
    return "Unknown"
```

This rule-based classifier performs a fast, deterministic lookup by checking whether any of the predefined keywords appear in the surface name string.

Examples:

* `"ABS-CBN News"` → `"Philippines"`
* `"BBC World"` → `"United Kingdom"`
* `"Wisconsin State Journal"` → `"United States"`

Keyword matching often provides the **strongest and highest-confidence signal**, especially for media entities and geo-localized page names.



In [24]:
## NER Location Extraction

def ner_location(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]


## Embedding similarity model

COUNTRY_LIST = list(COUNTRY_KEYWORDS.keys())
country_emb = model.encode(COUNTRY_LIST, convert_to_tensor=True)

def embedding_country(text):
    text_emb = model.encode(text, convert_to_tensor=True)
    scores = util.cos_sim(text_emb, country_emb)[0]
    best_idx = scores.argmax().item()
    return COUNTRY_LIST[best_idx], float(scores[best_idx])


## Keywords matching

def keyword_country(text):
    txt = text.lower()
    for country, keywords in COUNTRY_KEYWORDS.items():
        for kw in keywords:
            if kw.lower() in txt:
                return country
    return "Unknown"

## Ensemble Country Inference Model

This cell defines the main function responsible for combining multiple independent signals into a single country prediction and confidence score for each surface name.

The model collects four types of signals:

1. **Keyword Match (Strongest Signal)**
   If a surface name contains known country-specific keywords, this contributes the highest weight. It is the most reliable indicator of origin for media outlets, organizations, and geographically named pages.

2. **Location Extraction via NER**
   If the surface name contains identifiable geopolitical entities (such as cities or regions), these are converted into country predictions using embedding similarity. This provides a strong geographic cue when explicit country keywords are missing.

3. **Language-Based Signals**
   The detected language of the text (e.g., English, German, Chinese, Japanese) provides helpful contextual clues. These signals carry modest weight since language alone is not definitive.

4. **Embedding Similarity (Fallback Signal)**
   When no other clues are present, the model uses semantic similarity between the surface name text and country names to assign a weak but meaningful prediction.

After scoring all possible countries based on these inputs, the function selects the country with the highest cumulative score and returns it along with a normalized confidence value.

Overall, this ensemble approach allows the model to infer country-of-origin even in noisy, multilingual, or partially corrupted surface names commonly found in Meta Content Library datasets.


In [25]:
## Ensemble inference model

def infer_country(text):

    lang = detect_language(text)
    ner_locs = ner_location(text)
    key_match = keyword_country(text)
    embed_guess, embed_score = embedding_country(text)

    scores = {}

    # --- Keyword signal (strongest)
    if key_match != "Unknown":
        scores[key_match] = scores.get(key_match, 0) + 0.40

    # --- NER location → embedding country
    for loc in ner_locs:
        country, _ = embedding_country(loc)
        scores[country] = scores.get(country, 0) + 0.30

    # --- Language-based signals
    if lang.startswith("en"):
        scores["United States"] = scores.get("United States", 0) + 0.05
        scores["United Kingdom"] = scores.get("United Kingdom", 0) + 0.05
    if lang == "de":
        scores["Germany"] = scores.get("Germany", 0) + 0.10
    if lang.startswith("zh"):
        scores["China"] = scores.get("China", 0) + 0.10
    if lang == "ja":
        scores["Japan"] = scores.get("Japan", 0) + 0.10
    if lang == "ko":
        scores["South Korea"] = scores.get("South Korea", 0) + 0.10

    # --- Embedding similarity (weakest)
    scores[embed_guess] = scores.get(embed_guess, 0) + 0.05

    # Final decision
    if not scores:
        return "Unknown", 0.0

    best = max(scores, key=scores.get)
    return best, round(scores[best], 3)


In [26]:
## Run inference on all names

results = []
for text in names:
    country, conf = infer_country(text)
    results.append({
        "surface.name": text,
        "country": country,
        "confidence": conf
    })

df_inferred_country = pd.DataFrame(results)
df_inferred_country.to_csv("./data/inferred_countries_output.csv", index=False)

print("DONE! Output saved to inferred_countries_output.csv")


DONE! Output saved to inferred_countries_output.csv


## Merge Inferred Country Metadata into Main Dataset

This cell joins the country-inference results back into the full Meta Content Library dataset. The steps performed here are:

### **1. Rename columns for clarity**

The `country` and `confidence` columns from the inference output are renamed to:

* `inferred_country`
* `inferred_country_confidence`
  This prevents naming conflicts and makes downstream analysis more explicit.

### **2. Merge on `surface.name`**

The enriched inference dataframe is merged with the main MCL dataframe using a **left join** on the `surface.name` field.
This ensures:

* Every original record is preserved
* Inferred fields are added wherever a match exists
* Records with unseen or unmatched surface names will simply receive `NaN` in the new columns

### **3. Save the merged dataset**

The final, enriched dataframe—now containing country predictions and confidence scores—is optionally written to disk for further exploration, modeling, or reporting.

This step completes the enrichment pipeline, producing a dataset where each MCL content item is paired with its most likely country of origin.


In [27]:
# Load the inferred countries file
df_inferred_country = pd.read_csv("./data/inferred_countries_output.csv")

# Rename columns in inferred df for clarity BEFORE merge
df_inferred_country = df_inferred_country.rename(columns={
    "country": "inferred_country",
    "confidence": "inferred_country_confidence"
})

# Perform the merge on surface.name
df_merged = df_main.merge(
    df_inferred_country[["surface.name", "inferred_country", "inferred_country_confidence"]],
    on="surface.name",
    how="left"
)

# Save the merged dataframe (optional)
df_merged.to_csv("./data/merged_output_with_inferred_country.csv", index=False)

df_merged.head()

,activities,content_type,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,...,statistics.views_date_last_refreshed,statistics.wow_count,surface.id,surface.name,surface.type,surface.username,text,clean_text,inferred_country,inferred_country_confidence
0,NaN,status,2025-11-10T19:00:19+00:00,2980185815499999,False,en,NaN,NaN,NaN,NaN,...,NaN,3.0,3.362899e+15,The American Tribune,page,amtribnews,"Despite Owners Exhausting Legal Appeals, The C...","despite owners exhausting legal appeals, the c...",United States,0.50
1,NaN,albums,2025-11-10T18:23:30+00:00,686591307858179,False,ro,NaN,NaN,NaN,NaN,...,NaN,0.0,2.810224e+13,Lache Flausat,page,IShouldBeSoLacheLacheLachee,Convorbiri literare 1. Fata noastra lasconista...,convorbiri literare 1. fata noastra lasconista...,Romania,0.40
2,NaN,videos,2025-11-10T18:14:12+00:00,1006177595025072,False,en,NaN,NaN,NaN,NaN,...,NaN,0.0,7.146830e+14,Maria Gerke,profile,maria.gerke,"Is this the face of evil? Apparently, this man...","is this the face of evil? apparently, this man...",Italy,0.05
3,NaN,links,2025-11-10T16:24:10+00:00,840171035327111,False,en,brecon-radnor.co.uk,NaN,https://www.brecon-radnor.co.uk/news/farming/a...,Avian influenza confirmed at Powys premises,...,NaN,0.0,3.012233e+14,The Brecon and Radnor Express,page,BreconRadnorExpress,A case of Highly Pathogenic Avian Influenza (H...,a case of highly pathogenic avian influenza (h...,Spain,0.30
4,NaN,links,2025-11-10T16:05:06+00:00,1028189766103347,False,de,promisundmehr.de,"Promis, Prominente, Stars und Sternchen ... Di...",https://www.promisundmehr.de/vogelgrippe-bei-h...,Vogelgrippe bei Haustieren: Katzen sind potenz...,...,NaN,0.0,9.454473e+14,Promis und mehr,page,promisundmehr,Diese Symptome zeigen Katzen bei Vogelgrippe\n...,diese symptome zeigen katzen bei vogelgrippe\n...,Germany,0.50
